*  DSC530 Data Exploration and Analysis
*  Weeks 1 & 2 Coding Assignment
*  Ander (Trevor) McKee

# Chapter 3, Exercise 1

We want to look at data for the Facebook, Apple, Amazon, Netflix, and Google (FAANG) stocks, but we were given each as a separate CSV file (obtained using the stock_analysis package we will build in Chapter 7, Financial Analysis – Bitcoin and the Stock Market). Combine them into a single file and store the dataframe of the FAANG data as faang for the rest of the exercises:


a. Read in the aapl.csv, amzn.csv, fb.csv, goog.csv, and nflx.csv files.
b. Add a column to each dataframe, called ticker, indicating the ticker symbol it is for (Apple's is AAPL, for example); this is how you look up a stock. In this case, the filenames happen to be the ticker symbols.
c. Append them together into a single dataframe.
d. Save the result in a CSV file called faang.csv

In [25]:
# 1a. Read in the aapl.csv, amzn.csv, fb.csv, goog.csv, and nflx.csv files.

import pandas as pd
from pathlib import Path

# Cleaner than only string paths.
DATA_DIR = Path("exercises")

tickers = ["aapl", "amzn", "fb", "goog", "nflx"]

# Create an empty dictionary.
stock_data = {}

# Dynamically find the file path and populate dictionary.
for ticker in tickers:
    stock_data[ticker] = pd.read_csv(DATA_DIR / f"{ticker}.csv")

# The below code is not necessary,
# but I include it here to check each file.
    
# Create a list to store info about the files.
summary = []

for ticker, df in stock_data.items():
    summary.append({
        
        # Capitalize.
        "ticker": ticker.upper(),
        
        # Count rows.
        "rows": df.shape[0],
        
        # Count columns.
        "columns": df.shape[1],
        
        # Return list of column names.
        "column_names": list(df.columns)
    })

    # Print summary.
pd.DataFrame(summary)

,ticker,rows,columns,column_names
0,AAPL,251,6,"[date, high, low, open, close, volume]"
1,AMZN,251,6,"[date, high, low, open, close, volume]"
2,FB,251,6,"[date, high, low, open, close, volume]"
3,GOOG,251,6,"[date, high, low, open, close, volume]"
4,NFLX,251,6,"[date, high, low, open, close, volume]"


In [26]:
# 1b. Add a column to each dataframe, called ticker, indicating the ticker symbol it is for (Apple's is AAPL, for example);
# this is how you look up a stock. In this case, the filenames happen to be the ticker symbols.

# Start a loop through the dictionary
for ticker, df in stock_data.items():
    
    # Fill columns with ticker.
    df["ticker"] = ticker.upper()

# Check first few rows for success.
stock_data["aapl"].head()

,date,high,low,open,close,volume,ticker
0,2018-01-02,43.075001,42.314999,42.540001,43.064999,102223600.0,AAPL
1,2018-01-03,43.637501,42.990002,43.132500,43.057499,118071600.0,AAPL
2,2018-01-04,43.367500,43.020000,43.134998,43.257500,89738400.0,AAPL
3,2018-01-05,43.842499,43.262501,43.360001,43.750000,94640000.0,AAPL
4,2018-01-08,43.902500,43.482498,43.587502,43.587502,82271200.0,AAPL


In [27]:
# 1c. Append them together into a single dataframe.

# Concat the files together.
faang = pd.concat(stock_data.values(), ignore_index=True)

# Expected shape is 5 files x 251 rows = 1,255 rows
# and 6 original columns + ticker = 7 columns.
print(f"Shape: {faang.shape}")

Shape: (1255, 7)


In [28]:
# 1d. Save the result in a CSV file called faang.csv,
# reset indexes, and check for file.

# Save to CSV.
faang.to_csv("faang.csv", index=False)

# Check the file exists.
Path("faang.csv").exists()

True

# Chapter 3, Exercise 2

With faang, use type conversion to cast the values of the date column into datetimes and the volume column into integers. Then, sort by date and ticker.

In [29]:
faang = (
    faang
    .assign(
        # Convert date strings to datetime values.
        date=lambda df: pd.to_datetime(df["date"], errors="raise"),
        
        # Convert volume values to integers.
        volume=lambda df: pd.to_numeric(df["volume"], errors="raise").astype("int64")
    )
    
    # Sort.
    .sort_values(["date", "ticker"])
    
    # Drop the old indexes and start fresh.
    .reset_index(drop=True)
)

# Check types.
faang.dtypes

date      datetime64[ns]
high             float64
low              float64
open             float64
close            float64
volume             int64
ticker            object
dtype: object

# Chapter 3, Exercise 3

Find the seven rows in faang with the lowest value for volume.

In [30]:
# .nsmallest is the most direct way here,
# because there is little use in sorting
# the entire df to display only the top 7.
lowest_volume = faang.nsmallest(7, "volume")

# Display.
lowest_volume

,date,high,low,open,close,volume,ticker
633,2018-07-03,1135.819946,1100.020020,1135.819946,1102.890015,679000,GOOG
1133,2018-11-23,1037.589966,1022.398987,1030.000000,1023.880005,691500,GOOG
498,2018-05-24,1080.469971,1066.150024,1079.000000,1079.239990,766800,GOOG
653,2018-07-10,1159.589966,1149.589966,1156.979980,1152.839966,798400,GOOG
763,2018-08-09,1255.541992,1246.010010,1249.900024,1249.099976,848600,GOOG
798,2018-08-20,1211.000000,1194.625977,1205.020020,1207.770020,870800,GOOG
808,2018-08-22,1211.839966,1199.000000,1200.000000,1207.329956,887400,GOOG


# Chapter 3, Exercise 4

Right now, the data is somewhere between long and wide format. Use melt() to make it completely long format. Hint: date and ticker are our ID variables (they uniquely identify each row). We need to melt the rest so that we don't have separate columns for open, high, low, close, and volume.

In [31]:
value_columns = ["open", "high", "low", "close", "volume"]

faang_long = faang.melt(
    
    # Leave date and ticker alone.
    id_vars=["date", "ticker"],
    
    # Melt these columns.
    value_vars=value_columns,
    
    # Name column for measurements.
    var_name="measurement",
    
    # Name column for values.
    value_name="value"
)

# Check first few rows.
faang_long.head()

,date,ticker,measurement,value
0,2018-01-02,AAPL,open,42.540001
1,2018-01-02,AMZN,open,1172.000000
2,2018-01-02,FB,open,177.679993
3,2018-01-02,GOOG,open,1048.339966
4,2018-01-02,NFLX,open,196.100006


# Chapter 3, Exercise 5

Suppose we found out that on July 26, 2018 there was a glitch in how the data was recorded. How should we handle this? Note that there is no coding required for this exercise.

To start, we shouldn't treat the data that was collected for July 26, 2018 as reliable. If there is a trustworthy source for the data, we could replace the glitched data with the corrected ones. In the more likely case they were not available, we might remove the affected records entirely or mark them as missing. Analyses that can handle missing values suggest they should be marked missing, and analyses that can't suggest they should be removed.

Either way, the records cannot be left alone as they appear in the data, as they could mislead summary statistics or visualizations. There might be a case to impute the values if we could be reasonably certain about them. In any case, we would have to document how we handled the values, so those reading our analysis have the context to understand what was going on in the data prior to manipulation.

My personal decision would likely be to remove those records. Some analysis technniques cannot handle missing values, and there are plenty of other values to use for analysis in the data.

# Chapter 3, Exercise 6

The European Centre for Disease Prevention and Control (ECDC) provides an open dataset on COVID-19 cases called daily number of new reported cases of COVID-19 by country worldwide. This dataset is updated daily, but we will use a snapshot that contains data from January 1, 2020 through September 18, 2020. Clean and pivot the data so that it is in wide format:


a. Read in the covid19_cases.csv file.
b. Create a date column using the data in the dateRep column and the pd.to_datetime() function.
c. Set the date column as the index and sort the index.
d. Replace all occurrences of United_States_of_America and United_Kingdom with USA and UK, respectively. Hint: the replace() method can be run on the dataframe as a whole.
e. Using the countriesAndTerritories column, filter the cleaned COVID-19 cases data down to Argentina, Brazil, China, Colombia, India, Italy, Mexico, Peru, Russia, Spain, Turkey, the UK, and the USA.
f. Pivot the data so that the index contains the dates, the columns contain the country names, and the values are the case counts (the cases column). Be sure to fill in NaN values with 0.

In [32]:
# a. Read in data.
covid = pd.read_csv(DATA_DIR / "covid19_cases.csv")

# Create list of countries.
countries = [
    "Argentina", "Brazil", "China", "Colombia", "India", "Italy",
    "Mexico", "Peru", "Russia", "Spain", "Turkey", "UK", "USA"
]

covid_clean = (
    
    # Start with the original data.
    covid
    .assign(
        
        # b. Convert dates to datetime values.
        date=lambda df: pd.to_datetime(df["dateRep"], format="%d/%m/%Y", errors="raise"), # Raise an error if can't be converted.
        
        # d. Replace long country names with abbreviations.
        countriesAndTerritories=lambda df: df["countriesAndTerritories"].replace({
            "United_States_of_America": "USA",
            "United_Kingdom": "UK"
        })
    )
    
    # e. Check which countries are in list and keep only True values.
    .loc[lambda df: df["countriesAndTerritories"].isin(countries)]
)

covid_wide = (
    
    # Used cleaned data.
    covid_clean
    
    # f. Convert the data into wide format.
    .pivot_table(
        
        # c1. Use date as index.
        index="date",
        
        # Make columns the countries and territories.
        columns="countriesAndTerritories",
        
        # Fill cells with cases.
        values="cases",
        
        # If more than one value, sum them.
        aggfunc="sum",
        
        # If no value, insert 0.
        fill_value=0
    )
    
    # c2. Sort by date.
    .sort_index()
)

# Check first few rows.
covid_wide.head()

countriesAndTerritories,Argentina,Brazil,China,Colombia,India,Italy,Mexico,Peru,Russia,Spain,Turkey,UK,USA
date,,,,,,,,,,,,,
2020-01-01,0,0,0,0,0,0,0,0,0,0,0,0,0
2020-01-02,0,0,0,0,0,0,0,0,0,0,0,0,0
2020-01-03,0,0,17,0,0,0,0,0,0,0,0,0,0
2020-01-04,0,0,0,0,0,0,0,0,0,0,0,0,0
2020-01-05,0,0,15,0,0,0,0,0,0,0,0,0,0
